In [ ]:
import os
from typing import Literal

from google import genai
from pydantic import BaseModel, Field


# ---------------------------------------------------------
# 1. Define the schema we want Gemini to return
# ---------------------------------------------------------

In [ ]:
class SentimentAnalysis(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="Overall sentiment of the text."
    )

    confidence: float = Field(
        description="Confidence score between 0.0 and 1.0."
    )

    summary: str = Field(
        description="A short summary explaining the sentiment."
    )

    key_points: list[str] = Field(
        description="Important points that influenced the sentiment."
    )

# ---------------------------------------------------------
# 2. Create Gemini client
# ---------------------------------------------------------

In [ ]:
client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

# ---------------------------------------------------------
# 3. Input text
# ---------------------------------------------------------

In [ ]:
text = """
I recently purchased these wireless headphones and I am extremely
happy with them. The sound quality is excellent, the battery lasts
all day, and they are very comfortable to wear.

The only downside is that the Bluetooth connection occasionally
drops, but reconnecting is quick and easy.

Overall, this is one of the best headphones I have owned.
"""

# ---------------------------------------------------------
# 4. Ask Gemini to return structured JSON
# ---------------------------------------------------------

In [ ]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",

    contents=f"""
Analyze the sentiment of the following customer review.

Customer review:
{text}
""",

    config={
        # Tell Gemini that the response must be JSON
        "response_mime_type": "application/json",

        # Tell Gemini exactly what JSON structure to produce
        "response_schema": SentimentAnalysis,
    },
)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


# ---------------------------------------------------------
# 5. Get the structured result
# ---------------------------------------------------------

In [ ]:
result = response.parsed

# ---------------------------------------------------------
# 6. Print the result
# ---------------------------------------------------------

In [ ]:
print("Structured result:")
print(result)


print("\nIndividual fields:")
print("Sentiment  :", result.sentiment)
print("Confidence :", result.confidence)
print("Summary    :", result.summary)

print("\nKey points:")
for point in result.key_points:
    print("-", point)